# Pipeline 4 — Social Post → Donation Value Predictor

**Created:** 2026-04-06T23:33:27.173943Z

This notebook follows **CRISP-DM** while also satisfying the IS455 rubric sections:
- Problem Framing
- Data Acquisition, Preparation & Exploration
- Modeling & Feature Selection
- Evaluation & Interpretation
- Causal and Relationship Analysis
- Deployment Notes


## CRISP-DM Overview

### 1) Business Understanding
Goal: help leaders post strategically by predicting expected donation value from post characteristics.

### 2) Data Understanding
Use social_media_posts table. Target is estimated_donation_value_php.

### 3) Data Preparation
Separate actionable pre-post features from post-performance engagement features (explanatory).

### 4) Modeling
Predictive: Gradient Boosting Regressor (pre-post features). Explanatory: Linear Regression with engagement metrics.

### 5) Evaluation
Evaluate MAE/RMSE; discuss what is controllable vs observed after posting.

### 6) Deployment
Export expected donation value per post; use results to guide posting guidelines.


## 1) Problem Framing (Rubric)

State:
- the business question,
- who cares,
- why it matters,
- predictive vs explanatory goals.

We build **two models**:
- Predictive (optimize out-of-sample performance)
- Explanatory (interpretability / relationship analysis)


## 2) Data Acquisition, Preparation & Exploration (Rubric)

Rules to avoid leakage:
- Define an **as-of date** (cutoff).
- Build features using only data **on or before** the cutoff.
- Create labels using only data **after** the cutoff in a defined horizon.


In [ ]:
import json
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor


REPO_ROOT = Path("..").resolve()
RAW_DIR_A = (REPO_ROOT / "data" / "raw" / "lighthouse_csv_v7").resolve()
RAW_DIR_B = (REPO_ROOT / "data" / "raw").resolve()
DATA_DIR = RAW_DIR_A if RAW_DIR_A.exists() else RAW_DIR_B

OUT_DIR = (REPO_ROOT / "output" / "ml-predictions").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Out dir:", OUT_DIR)


def require_csv(stem: str) -> pd.DataFrame:
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}.")
    return pd.read_csv(path, encoding="utf-8-sig")


# After POST /api/admin/lighthouse-import, train from Azure SQL by setting INTEX_ODBC to your ODBC connection string.
SQL_TABLE_BY_STEM = {
    "supporters": "Supporters",
    "donations": "Contributions",
    "social_media_posts": "SocialMediaPosts",
    "safehouse_monthly_metrics": "SafehouseMonthlyMetrics",
    "residents": "Residents",
    "incident_reports": "IncidentReports",
    "home_visitations": "HomeVisitations",
    "process_recordings": "ProcessRecordings",
    "education_records": "EducationRecords",
    "health_wellbeing_records": "HealthWellbeingRecords",
}


def load_df(stem: str) -> pd.DataFrame:
    odbc = os.environ.get("INTEX_ODBC")
    table = SQL_TABLE_BY_STEM.get(stem)
    if odbc and table:
        try:
            import pyodbc

            with pyodbc.connect(odbc, timeout=120) as cnx:
                df = pd.read_sql(f"SELECT * FROM [{table}]", cnx)
            print(f"DB [{table}] rows:", len(df))
            if len(df) > 0:

                def pascal_to_snake(name: str) -> str:
                    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

                df = df.rename(columns={c: pascal_to_snake(str(c)) for c in df.columns})
                if stem == "donations":
                    df = df.rename(columns={"contribution_id": "donation_id"})
                if stem == "process_recordings":
                    df = df.rename(columns={"process_recording_id": "recording_id"})
                if stem == "home_visitations":
                    df = df.rename(columns={"home_visitation_id": "visitation_id"})
                return df
        except Exception as ex:
            print("INTEX_ODBC failed, using CSV:", ex)
    return require_csv(stem)


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.date


def to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)


def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out


def eval_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def export_predictions_json(prediction_type: str, entity_type: str, df_out: pd.DataFrame, id_col: str, score_col: str, label_col: str | None = None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    rows = []
    for _, r in df_out.iterrows():
        rows.append(
            {
                "predictionType": prediction_type,
                "entityType": entity_type,
                "entityId": int(r[id_col]),
                "score": float(r[score_col]),
                "label": None if label_col is None else (None if pd.isna(r[label_col]) else str(r[label_col])),
                "payloadJson": json.dumps({k: v for k, v in r.items() if k not in {id_col, score_col, label_col}}, default=str),
            }
        )
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Wrote:", out_path, "rows=", len(rows))


In [ ]:
posts = load_df("social_media_posts")
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce")
posts = posts.dropna(subset=["created_at"]).sort_values("created_at").copy()

posts["estimated_donation_value_php"] = pd.to_numeric(posts["estimated_donation_value_php"], errors="coerce").fillna(0)
posts["donation_referrals"] = pd.to_numeric(posts["donation_referrals"], errors="coerce").fillna(0)

# Predictive features available at creation time (actionable)
pre_features = [
    "platform","day_of_week","post_hour","post_type","media_type",
    "num_hashtags","mentions_count","has_call_to_action","call_to_action_type",
    "content_topic","sentiment_tone","caption_length","features_resident_story",
    "campaign_name","is_boosted","boost_budget_php"
]
for c in ["num_hashtags","mentions_count","caption_length","boost_budget_php","post_hour"]:
    posts[c] = pd.to_numeric(posts[c], errors="coerce").fillna(0)

# Explanatory features (includes engagement metrics; not available before posting)
exp_features = pre_features + [
    "impressions","reach","likes","comments","shares","saves","click_throughs","video_views",
    "engagement_rate","profile_visits","follower_count_at_post","forwards"
]
for c in ["impressions","reach","likes","comments","shares","saves","click_throughs","video_views",
          "engagement_rate","profile_visits","follower_count_at_post","forwards"]:
    posts[c] = pd.to_numeric(posts[c], errors="coerce").fillna(0)

target = "estimated_donation_value_php"

# Time split
cutoff = posts["created_at"].quantile(0.8)
train = posts[posts["created_at"] <= cutoff].copy()
test = posts[posts["created_at"] > cutoff].copy()
print("Train:", len(train), "Test:", len(test), "Cutoff:", cutoff)


In [ ]:
def fit_regression(df_train, df_test, features, model):
    cat_cols = [c for c in features if df_train[c].dtype == "object" or c in ["platform","day_of_week","post_type","media_type","call_to_action_type","content_topic","sentiment_tone","campaign_name"]]
    num_cols = [c for c in features if c not in cat_cols]
    pre = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("num", "passthrough", num_cols)]
    )
    pipe = Pipeline(steps=[("pre", pre), ("model", model)])
    pipe.fit(df_train[features], df_train[target])
    pred = pipe.predict(df_test[features])
    return pipe, pred

# Predictive (actionable) model
pred_model, pred_hat = fit_regression(train, test, pre_features, GradientBoostingRegressor(random_state=42))
print("Predictive (pre-post features):", eval_regression(test[target], pred_hat))

# Explanatory model (uses engagement metrics)
exp_model, exp_hat = fit_regression(train, test, exp_features, LinearRegression())
print("Explanatory (includes engagement):", eval_regression(test[target], exp_hat))


In [ ]:
# Score each post (what we would have expected based on the plan)
posts_out = posts[["post_id"] + pre_features].copy()
posts_out["predicted_value_php"] = pred_model.predict(posts_out[pre_features])
posts_out["value_band"] = pd.qcut(posts_out["predicted_value_php"].rank(method="first"), q=4, labels=["Low","Medium","High","Very High"])

export_predictions_json(
    prediction_type="post_donation_value",
    entity_type="SocialPost",
    df_out=posts_out[["post_id","predicted_value_php","value_band","platform","post_type","media_type","post_hour","has_call_to_action","is_boosted","boost_budget_php"]].rename(columns={"post_id":"post_id","predicted_value_php":"score"}),
    id_col="post_id",
    score_col="score",
    label_col="value_band"
)


## 3) Modeling & Feature Selection (Rubric)

- Predictive model: tree/ensemble
- Explanatory model: linear/logistic regression


## 4) Evaluation & Interpretation (Rubric)

Interpret in business terms, and discuss real-world costs of errors.

## 5) Causal and Relationship Analysis (Rubric)

Discuss relationships, confounding risks, and where correlation ≠ causation.


## 6) Deployment Notes (Rubric)

Export predictions to JSON and import into the deployed app:
- `POST /api/ml/import?replace=true` (admin-only)
- View in `/app/ml` (Staff Portal → ML Insights)
